# PhasePhyto Report Viewer

Displays text reports for one saved run, especially the PlantDoc/target classification report and manifest summary.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    # Set this to a specific run folder name under drive_project_dir/runs.
    # If None, the latest run by folder name is used.
    "run_name": None,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + LOCATE RUN
# ============================================================

from pathlib import Path
import json

from google.colab import drive
from IPython.display import Image, Markdown, display

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path(CONFIG["drive_project_dir"])
RUNS_DIR = PROJECT_DIR / "runs"

if not RUNS_DIR.exists():
    raise RuntimeError(f"No runs directory found: {RUNS_DIR}")

if CONFIG["run_name"]:
    RUN_DIR = RUNS_DIR / CONFIG["run_name"]
else:
    candidates = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No run directories found under {RUNS_DIR}")
    RUN_DIR = candidates[-1]

MANIFEST_PATH = RUN_DIR / "run_manifest.json"
if not MANIFEST_PATH.exists():
    raise RuntimeError(f"No run_manifest.json found at {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text())
artifacts = manifest.get("artifacts", {})

print(f"Loaded run: {RUN_DIR}")
print(f"Manifest: {MANIFEST_PATH}")


In [ ]:
# ============================================================
# SHOW TARGET CLASSIFICATION REPORT
# ============================================================

report_path = Path(artifacts.get("target_classification_report", RUN_DIR / "results" / "target_classification_report.txt"))
if report_path.exists():
    display(Markdown("## Target Classification Report"))
    print(report_path.read_text())
else:
    print(f"Missing classification report: {report_path}")


In [ ]:
# ============================================================
# SHOW COMPACT RUN SUMMARY
# ============================================================

summary = {
    "run_name": manifest.get("run_name"),
    "storage_backend": manifest.get("storage_backend"),
    "run_dir": manifest.get("run_dir"),
    "source_root": manifest.get("data", {}).get("source_root"),
    "target_root": manifest.get("data", {}).get("target_root"),
}
print(json.dumps(summary, indent=2))
